# POC: Instantiate Points object as pydantic class

As part of the future refactoring of geoh5py, this spike explores creating an entity as a pydantic class.

Goals:

- Direct access of attributes and data from the geoh5 dataset
- Validation of attributes as pydantic model fields
- Lazy loading of large arrays such as vertices, cells, and data
- Instantiation of the class without the need for a parent workspace

## Imports



In [14]:
import pickle
from uuid import uuid4

import numpy as np
from pydantic import ValidationError

from geoh5py_pydantic import CallableArraySource, PointsModel

## Instantiate without a workspace

Create a Points-like entity directly from attributes and vertices without creating a Workspace.

In [15]:
# PointsModel can be created directly from plain coordinates.
example_array = np.array(
    [
        [0.0, 0.0, 0.0],
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ]
)

points = PointsModel(
    name="Standalone points",
    vertices=example_array,
)

# Can pickle the PointsModel:

with open("points.pkl", "wb") as file:
    pickle.dump(points, file)

with open("points.pkl", "rb") as file:
    points_loaded = pickle.load(file)

points_loaded

PointsModel(uid=UUID('d569229d-c54f-4f34-b2c5-098e16e22b94'), name='Standalone points', type_uid=UUID('202c5db1-a56d-4004-9cad-baafd8899406'), parent_uid=None, allow_delete=True, allow_move=True, allow_rename=True, clipping_ids=None, metadata=None, on_file=False, partially_hidden=False, public=True, visible=True, last_focus='None', vertices=array([[0., 0., 0.],
       [1., 2., 3.],
       [4., 5., 6.]]))

In [16]:
points

PointsModel(uid=UUID('d569229d-c54f-4f34-b2c5-098e16e22b94'), name='Standalone points', type_uid=UUID('202c5db1-a56d-4004-9cad-baafd8899406'), parent_uid=None, allow_delete=True, allow_move=True, allow_rename=True, clipping_ids=None, metadata=None, on_file=False, partially_hidden=False, public=True, visible=True, last_focus='None', vertices=array([[0., 0., 0.],
       [1., 2., 3.],
       [4., 5., 6.]]))

In [17]:
# Entity-style attributes and computed geometry are available directly.
points.uid, points.name, points.n_vertices

(UUID('d569229d-c54f-4f34-b2c5-098e16e22b94'), 'Standalone points', 3)

In [18]:
# Vertices are exposed as a normal (n, 3) float array for convenient use.
points.vertices_array

array([[0., 0., 0.],
       [1., 2., 3.],
       [4., 5., 6.]])

In [19]:
# Extent is derived from the vertex coordinates.
points.extent

array([[0., 0., 0.],
       [4., 5., 6.]])

## Assignment validation

Pydantic assignment validation means changing model fields should rerun the same validation rules.

In [20]:
# Replacing vertices with another valid array succeeds and updates derived values.
points.vertices = np.array([[10.0, 11.0, 12.0], [13.0, 14.0, 15.0]])
points.vertices_array

array([[10., 11., 12.],
       [13., 14., 15.]])

In [21]:
# Invalid shapes cause pydantic ValidationError messages.
try:
    PointsModel(vertices=np.r_[1.0, 2.0, 3.0])
except ValidationError as error:
    print(error)

1 validation error for PointsModel
vertices
  Value error, Array of 'vertices' should be of shape (*, 3). Got shape (3,). [type=value_error, input_value=array([1., 2., 3.]), input_type=ndarray]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


## Lazy array

This simulates a future geoh5 adapter where large arrays are not loaded until code actually asks for them.

In [22]:
# CallableArraySource simulates some kind of IO adapter that fetches vertices by uid/key.
# PointsModel should be able to represent a Points object but delay loading the actual vertex array until later
uid = uuid4()
calls = []


def fetcher(entity_uid, key):
    # if asked for an array belonging to entity_uid and named key,
    # record that this was called then return this fake vertices array
    calls.append((str(entity_uid), key))
    return np.array(
        [
            [100.0, 200.0, 300.0],
            [101.0, 201.0, 301.0],
        ]
    )


source = CallableArraySource(fetcher)

lazy_points = PointsModel.from_array_source(source, uid, name="Lazy points")

lazy_points.vertices, lazy_points.vertices.is_loaded, calls
# Should show that the vertices array is not loaded yet, and the fetcher has not been called.

(LazyArray(key='vertices', uid=ca3720af-cd59-4d12-bc41-4cc6360da3ef, state=lazy),
 False,
 [])

In [23]:
# Accessing n_vertices forces the LazyArray to load and validate its data.
lazy_points.n_vertices, lazy_points.vertices.is_loaded, calls

(2, True, [('ca3720af-cd59-4d12-bc41-4cc6360da3ef', 'vertices')])

In [24]:
# Subsequent access reuses the cached loaded array.
lazy_points.vertices_array

array([[100., 200., 300.],
       [101., 201., 301.]])

## geoh5-style output



In [25]:
# Dump core attributes using geoh5 names and convert vertices to the geoh5 dtype.
attrs = lazy_points.model_dump_geoh5_attributes()
vertices = attrs.pop("Vertices")

sorted(attrs), vertices.dtype, vertices.shape, vertices.view("<f8").reshape((-1, 3))

(['Allow delete',
  'Allow move',
  'Allow rename',
  'Clipping IDs',
  'ID',
  'Last focus',
  'Metadata',
  'Name',
  'Object Type ID',
  'Partially hidden',
  'Public',
  'Visible'],
 dtype((numpy.record, [('x', '<f8'), ('y', '<f8'), ('z', '<f8')])),
 (2,),
 array([[100., 200., 300.],
        [101., 201., 301.]]))

## Legacy adapter

A way for existing geoh5py Points objects to be adapted to the pydantic model.
If the plan is just to swap from the current geoh5py setup to this new version, this adapter probably won't be needed
though it could be useful for testing and comparison.

In [26]:
# Existing geoh5py Points can be adapted to the pydantic model for comparison.
from geoh5py.objects import Points
from geoh5py.workspace import Workspace


workspace = Workspace()
legacy = Points.create(
    workspace,
    name="Legacy points",
    vertices=np.array([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0]]),
)

adapted = PointsModel.from_legacy_points(legacy)
type(legacy), type(adapted), adapted.name, adapted.vertices_array

(geoh5py.objects.points.Points,
 geoh5py_pydantic.points.PointsModel,
 'Legacy points',
 array([[1., 1., 1.],
        [2., 2., 2.]]))